# Table maintenance

`OPTIMIZE` every table in the project's schemas, and optionally `VACUUM`.

Predictive optimization handles this automatically on managed Unity Catalog
tables in most workspaces; keep this job for the cases it does not cover, or as
the place to put a deliberate retention policy.

**VACUUM is destructive**: it removes files older than the retention threshold
and breaks time travel past that point. It is off unless `run_vacuum=True`.

In [0]:
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())

ENV = configs.get("env", "dev")
CATALOG_PREFIX = configs.get("catalog_prefix", "rearc")
RUN_VACUUM = configs.get("run_vacuum", "False").lower() == "true"

CATALOG = f"{CATALOG_PREFIX}_{ENV}"
SCHEMAS = ["bronze", "silver", "gold"]
VACUUM_RETAIN_HOURS = 168  # 7 days, the Delta default

print(f"catalog={CATALOG} | schemas={SCHEMAS} | vacuum={RUN_VACUUM}")

In [0]:
tables = []
for schema in SCHEMAS:
    rows = spark.sql(f"SHOW TABLES IN {CATALOG}.{schema}").collect()
    tables += [f"{CATALOG}.{schema}.{r.tableName}" for r in rows if not r.isTemporary]

print(f"{len(tables)} tables")
for t in tables:
    print(f"  {t}")

In [0]:
for table in tables:
    spark.sql(f"OPTIMIZE {table}")
    print(f"optimized {table}")

In [0]:
if RUN_VACUUM:
    for table in tables:
        spark.sql(f"VACUUM {table} RETAIN {VACUUM_RETAIN_HOURS} HOURS")
        print(f"vacuumed {table}")
else:
    print("vacuum skipped (set run_vacuum=True on the job to enable)")